# Prepare the data

In [ ]:
# Load the data
import pandas as pd
import numpy as np

data = pd.read_csv('../data/raw/dataset.csv', na_values=["", "NaN", "-", "null", "Null", "none", "None"])

# Drop NaNs and irrelevant columns, define features array and target variable
data = data.dropna()
data.drop_duplicates(inplace=True)

In [3]:
# Filter out all the samples that are copies of the original song
artist_med = data.groupby('artists')['popularity'].transform('median')
junk = (artist_med > 30) & (data['popularity'] < 10)

data = data[~junk].copy()

data["primary_artist"] = data["artists"].str.split(";").str[0].str.strip()
g = data.groupby(["primary_artist", "duration_ms"])
g.filter(lambda x: len(x) > 1).sort_values(["primary_artist", "duration_ms"])[
    ["track_name", "album_name", "popularity"]
].nunique()

track_name    15639
album_name    11603
popularity      100
dtype: int64

In [4]:
data[data['popularity'] == 0]

,track_id,spotify_track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,primary_artist
23,23,0BUuuEvNa5T4lMaewyiudB,Jason Mraz,Coffee Moment,93 Million Miles,0,216386,False,0.572,0.454,...,1,0.0258,0.4770,0.000014,0.0974,0.5150,140.182,4,acoustic,Jason Mraz
24,24,3Hn3LfhrQOaKihdCibJsTs,Jason Mraz,Human - Best Adult Pop Tunes,Unlonely,0,231266,False,0.796,0.667,...,0,0.0392,0.3810,0.000000,0.2210,0.7540,97.988,4,acoustic,Jason Mraz
26,26,5IfCZDRXZrqZSm8AwE44PG,Jason Mraz,Holly Jolly Christmas,Winter Wonderland,0,131760,False,0.620,0.309,...,1,0.0495,0.7880,0.000000,0.1460,0.6640,145.363,4,acoustic,Jason Mraz
27,27,0dzKBptH2P5j5a0MifBMwM,Jason Mraz,Feeling Good - Adult Pop Favorites,If It Kills Me,0,273653,False,0.633,0.429,...,0,0.0381,0.0444,0.000000,0.1320,0.5200,143.793,4,acoustic,Jason Mraz
28,28,5QAMZTM5cmLg3fHX9ZbTZi,Jason Mraz,Christmas Time,Winter Wonderland,0,131760,False,0.620,0.309,...,1,0.0495,0.7880,0.000000,0.1460,0.6640,145.363,4,acoustic,Jason Mraz
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113039,113039,5Z5nDrUm4et2SBtLDKnF38,Michael W. Smith;Lady Antebellum,Santa's Christmas List,White Christmas,0,229106,False,0.335,0.552,...,1,0.0357,0.4650,0.000001,0.1020,0.3500,126.622,4,world-music,Michael W. Smith
113042,113042,2eXCP4kLEm5b11TH2MVas6,Chris Tomlin;Audrey Assad,Santa's Christmas List,Winter Snow,0,212773,False,0.444,0.330,...,1,0.2620,0.9540,0.000001,0.6760,0.3080,95.470,4,world-music,Chris Tomlin
113043,113043,2XdGsYBDQYXjgdixMdn7w2,Matt Redman;Chris Tomlin,Slow Christmas Songs 2022,Angels (Singing Gloria),0,316920,False,0.424,0.239,...,1,0.0309,0.6580,0.000017,0.0951,0.0697,141.880,4,world-music,Matt Redman
113049,113049,6E7Ix5jkd6uzfoxuvcI8Ww,Rend Collective;We The Kingdom,Santa's Christmas List,God Rest Ye Merry Gentlemen (Hallelujah),0,217120,False,0.607,0.884,...,1,0.0489,0.0230,0.000000,0.2260,0.5550,139.988,4,world-music,Rend Collective


In [5]:
# Drop all the duplicate songs that have the same primary artists and duration
POP = "popularity"
ID  = "spotify_track_id"

data["primary_artist"] = data["artists"].str.split(";").str[0].str.strip()
data = data.sort_values(POP, ascending=False)        # keep="first" now == keep max popularity

data = data.drop_duplicates(["primary_artist", "duration_ms"], keep="first").reset_index(drop=True)
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 81707 entries, 0 to 81706
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   track_id          81707 non-null  int64  
 1   spotify_track_id  81707 non-null  str    
 2   artists           81707 non-null  str    
 3   album_name        81707 non-null  str    
 4   track_name        81707 non-null  str    
 5   popularity        81707 non-null  int64  
 6   duration_ms       81707 non-null  int64  
 7   explicit          81707 non-null  bool   
 8   danceability      81707 non-null  float64
 9   energy            81707 non-null  float64
 10  key               81707 non-null  int64  
 11  loudness          81707 non-null  float64
 12  mode              81707 non-null  int64  
 13  speechiness       81707 non-null  float64
 14  acousticness      81707 non-null  float64
 15  instrumentalness  81707 non-null  float64
 16  liveness          81707 non-null  float64
 17  vale

In [6]:
# Remove rubbish samples with extreme duration and tempo values
MAX_MS  = 10 * 60 * 1000      # 600_000 ms
MIN_BPM = 40

before = len(data)
junk = (data["duration_ms"] >= MAX_MS) | (data["tempo"] <= MIN_BPM) | (data["duration_ms"] <= 60000)
data = data[~junk].reset_index(drop=True)
print(f"dropped {before - len(data)} rows ({(before - len(data)) / before:.1%})")

dropped 1407 rows (1.7%)


In [7]:
# Add new temporary feature - normalized song name
import re

ARTIST_COL, NAME_COL = "artists", "track_name"  # match your schema

QUALIFIER = (r"remaster(?:ed)?|remix|live|acoustic|deluxe|expanded|version|edit|"
             r"edition|mix|mono|stereo|radio|instrumental|demo|bonus|re-?recorded|"
             r"anniversary|single version")

def norm_name(name):
    s = str(name).lower().replace("\u2019", "'")                      # unify ’ and '
    s = re.sub(r"\s*[\(\[]?\s*(?:feat|ft|featuring)\.?\s.*?(?:[\)\]]|$)", " ", s)
    s = re.sub(rf"\s*[\(\[][^\)\]]*(?:{QUALIFIER})[^\)\]]*[\)\]]", " ", s)  # parens w/ qualifier only
    s = re.sub(rf"\s*-\s*[^-]*(?:{QUALIFIER}).*$", " ", s)             # trailing " - Live" etc.
    s = re.sub(r"[^a-z0-9 ]", "", s)                                  # drop remaining punctuation
    return re.sub(r"\s+", " ", s).strip()

data["song_key"] = (data[ARTIST_COL].str.split(";").str[0].str.strip().str.lower()
                  + " :: " + data[NAME_COL].map(norm_name))

In [8]:
# Drop duplicates that have the same song_key
before = len(data)
data = data.drop_duplicates("song_key", keep="first").drop(columns="song_key").reset_index(drop=True)
print(f"dropped {before - len(data)} rows ({(before - len(data)) / before:.1%})")

dropped 6638 rows (8.3%)


In [9]:
# explode to one row per (track, individual artist); original index preserved
df = data[['artists', 'popularity']].copy()
df['artist'] = df['artists'].str.split(';')
ex = df.explode('artist')
ex['artist'] = ex['artist'].str.strip()

# per-row leave-one-out mean within each artist to filter trash data
def loo(s):
    vals = s.to_numpy()
    n = vals.size
    if n == 1:                       # single track -> no "other" tracks
        return np.array([np.nan])
    return np.array([np.quantile(np.delete(vals, i), 0.75) for i in range(n)])

ex['artist_loo'] = ex.groupby('artist')['popularity'].transform(loo)

# collapse back to track level: max across the track's artists (skips NaN)
fame = ex.groupby(level=0)['artist_loo'].max()
data['artist_fame_loo'] = fame

# tracks where every artist had only one song -> no signal -> neutral fill
data['artist_fame_loo'] = data['artist_fame_loo'].fillna(
    data['popularity'].median()
)

In [10]:
before = len(data)
junk = (data['artist_fame_loo'] > 2 * data['popularity'])
data = data[~junk].copy()
print(f"dropped {before - len(data)} rows ({(before - len(data)) / before:.1%})")

dropped 6854 rows (9.3%)


In [11]:
# Drop unnecessary columns
orig_data = data.copy()
data = data.drop(columns=["track_id", "spotify_track_id", "artists", "album_name", "track_name", "primary_artist"])

In [12]:
data.to_parquet('../data/interim/tracks_enriched.parquet')
orig_data.to_parquet('../data/interim/orig_data.parquet')

In [13]:
orig_data[orig_data['primary_artist'] == "Sam Smith"]

,track_id,spotify_track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,primary_artist,artist_fame_loo
0,81051,3nqQXoyQOWXiESFLlDF1hG,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),Unholy (feat. Kim Petras),100,156943,False,0.714,0.472,...,0.0864,0.013,0.000005,0.2660,0.238,131.121,4,pop,Sam Smith,76.75
59,20907,7795WJLVKJoAyVoOtCWqXN,Sam Smith,In The Lonely Hour,I'm Not The Only One,88,239316,False,0.677,0.485,...,0.0361,0.529,0.000020,0.0766,0.493,82.001,4,dance,Sam Smith,76.75
308,20150,1mXVgsBdtIVeCLJnSnmtdV,Sam Smith,The Thrill Of It All (Special Edition),Too Good At Goodbyes,83,201000,False,0.681,0.372,...,0.0432,0.640,0.000000,0.1690,0.476,91.873,4,dance,Sam Smith,76.75
846,20813,7t3Xdbufg7q2onVsR8RBdY,Sam Smith,Love Goes,Fire On Fire,78,246735,False,0.584,0.408,...,0.0461,0.476,0.000000,0.1800,0.333,115.129,4,dance,Sam Smith,76.75
